# Voz D gratis con Chatterbox · vídeos enteros de golpe

1. Menú **Entorno de ejecución → Cambiar tipo de entorno → T4 GPU** → Guardar.
2. Pulsa ▶️ en cada bloque, de arriba abajo.
3. Bloque 2: sube **un mp3 de la voz D** (10–20 s, sin música).
4. Bloque 3: sube el **guion en .txt**, una frase por línea (cada línea = un archivo `01.mp3`, `02.mp3`…).
5. Bloque 5: descarga el .zip y copia los mp3 a `audio/voz/es/<video>/`.

In [ ]:
# 1 · Instalar (2-3 min). Si pide reiniciar la sesión: reinicia y sigue en el bloque 2
!pip install -q chatterbox-tts

In [ ]:
# 2 · Subir la voz de ejemplo (mp3 de la voz D)
from google.colab import files
EJEMPLO = list(files.upload())[0]
print('Voz de ejemplo:', EJEMPLO)

In [ ]:
# 3 · Subir el guion (.txt, una frase por línea; las líneas vacías se saltan)
GUION = list(files.upload())[0]
FRASES = [l.strip() for l in open(GUION, encoding='utf-8') if l.strip()]
print(len(FRASES), 'frases')

# Ajustes: si suena muy rápido baja RITMO a 0.3; para más emoción sube EMOCION a 0.7
EMOCION = 0.5
RITMO = 0.5

In [ ]:
# 4 · Generar. Guarda cada frase al terminarla: si Colab se corta, vuelve a pulsar ▶️ y sigue donde iba
import os, re, torch, torchaudio as ta
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
modelo = ChatterboxMultilingualTTS.from_pretrained(device='cuda', t3_model='v3')
os.makedirs('voces', exist_ok=True)
for i, f in enumerate(FRASES, 1):
    sal = f'voces/{i:02d}.mp3'
    if os.path.exists(sal): continue
    # las frases muy largas se parten por puntos y comas para que no se trabe
    trozos = [t for t in re.split(r'(?<=[.;:!?])\s+', f) if t] if len(f) > 250 else [f]
    wav = torch.cat([modelo.generate(t, language_id='es', audio_prompt_path=EJEMPLO, exaggeration=EMOCION, cfg_weight=RITMO) for t in trozos], dim=-1)
    ta.save('tmp.wav', wav, modelo.sr)
    os.system(f'ffmpeg -v error -y -i tmp.wav -b:a 160k {sal}')
    print(f'✓ {i:02d}/{len(FRASES)}', f[:70])

In [ ]:
# 5 · Descargar todo en un zip
!cd voces && zip -q ../voces.zip *.mp3
files.download('voces.zip')